<a href="https://colab.research.google.com/github/chantika-maharani/ProyekAkhir_5C_Estimation-of-Obesity-Levels/blob/main/Notebook/02_praproses.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laporan Praproses Data untuk Pemodelan K-Means

Praproses data merupakan tahapan krusial sebelum menerapkan algoritma clustering seperti K-Means. Karena K-Means menggunakan perhitungan jarak geometris (*Euclidean Distance*), performa model sangat dipengaruhi oleh perbedaan skala fitur dan keberadaan data kategorikal. Berikut adalah alur praproses data terstruktur yang dirancang agar hasil pemodelan menjadi valid.

### Langkah 1: Impor Library dan Memisahkan Fitur Utama
Kami mengimpor modul `KMeans` dari `sklearn.cluster` untuk pemodelan clustering. Pada analisis tanpa pengawasan, variabel target asli (`NObeyesdad`) dikeluarkan dari proses klasterisasi, namun tetap disimpan secara terpisah sebagai referensi pembanding di akhir.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Load dataset
df = pd.read_csv('/content/data_bersih (1).csv')
target_column = df.columns[-1]

# Memisahkan fitur untuk clustering (tanpa menyertakan label target asli)
X = df.drop(columns=[target_column])
y_true = df[target_column] # Disimpan hanya untuk analisis validasi eksternal

### Langkah 2: Identifikasi Tipe Atribut (Numerik vs Kategorikal)
Atribut numerik dan kategorikal memerlukan penanganan yang berbeda. Kami mengelompokkan jenis kolom ini secara otomatis berdasarkan tipe datanya.

In [6]:
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Fitur Numerik: {num_cols}")
print(f"Fitur Kategorikal: {cat_cols}")

Fitur Numerik: ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
Fitur Kategorikal: ['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']


### Langkah 2: Konstruksi Pipeline Praproses Data
Karena K-Means mengandalkan jarak geometris, semua fitur harus berupa numerik berskala seragam:
1. **Numerik**: Diimputasi dengan *median* dan dinormalisasi menggunakan `StandardScaler`.
2. **Kategorikal**: Diimputasi dengan *modus* lalu dikonversi menjadi biner via `OneHotEncoder`.

In [7]:
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

# Melakukan praproses pada seluruh data X
X_preprocessed = preprocessor.fit_transform(X)
print(f"Dimensi data setelah dipraproses untuk K-Means: {X_preprocessed.shape}")

Dimensi data setelah dipraproses untuk K-Means: (2087, 31)


# Ringkasan Hasil Praproses Data

Berikut adalah ringkasan teknis dari tahapan praproses data pada dataset `/content/data_bersih (1).csv`:

1. **Pemisahan Fitur Utama**:
   * Variabel target asli (`NObeyesdad`) dikeluarkan dari dataset fitur utama agar algoritma clustering dapat berfokus memetakan data berdasarkan kemiripan karakteristik tanpa adanya intervensi dari label klasifikasi.

2. **Konstruksi Pipeline Praproses (Standardisasi Ketat)**:
   * **Fitur Numerik**: Diimputasi menggunakan metode *median* dan dinormalisasi menggunakan `StandardScaler` agar memiliki skala rata-rata 0 dan variansi 1. Ini sangat penting agar fitur dengan skala besar tidak mendominasi perhitungan jarak.
   * **Fitur Kategorikal**: Diimputasi dengan nilai *modus* (kategori paling sering muncul) dan ditransformasikan secara aman menjadi representasi biner menggunakan `OneHotEncoder`.

3. **Dimensi Akhir Fitur**:
   * Hasil transformasi praproses menghasilkan matriks data akhir berdimensi **(2087, 31)** yang siap digunakan untuk proses analisis klaster di file atau tahap terpisah.